## Imports & Config

We import 3 libraries:
- requests: sends messages to Claude API over the internet
- subprocess: runs terminal commands on our computer
- json: reads our config.json file into Python

Instead of hardcoding settings in the code, we load them from a 
separate config.json file. This way we can change settings without 
touching the code.

In [14]:
import requests
import subprocess
import json
import os
from dotenv import load_dotenv

# Load settings from config file
load_dotenv()
with open("config.json", "r") as f:
    config = json.load(f)

print("Config loaded!")
print(config)


Config loaded!
{'model': 'claude-haiku-4-5-20251001', 'max_tokens': 500, 'max_commands': 5, 'system_prompt': 'You are a helpful software engineering agent running on Windows. Use Windows commands like dir, echo %DATE%, wmic. You must ALWAYS reply with a JSON object in one of these two formats: {"type": "command", "content": "the command here"} or {"type": "answer", "content": "your answer here"}. Never reply with anything else. Only respond to software engineering topics.'}


## API Key & ask_claude Function

The API Key is a unique password that gives access to Claude AI. 
It must be included in every request we send.

The ask_claude function is responsible for communicating with Claude.
It sends the full conversation history and receives Claude's reply.

In this version, all settings (model, max_tokens, system prompt) 
are loaded automatically from the config.json file instead of 
being hardcoded in the code. This makes the agent easier to 
configure without changing the code itself.

In [15]:
API_KEY = os.getenv("ANTHROPIC_API_KEY")

def ask_claude(messages):
    response = requests.post(
        url="https://api.anthropic.com/v1/messages",
        headers={
            "x-api-key": API_KEY,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json"
        },
        json={
            "model": config["model"],
            "max_tokens": config["max_tokens"],
            "system": config["system_prompt"],
            "messages": messages
        }
    )
    return response.json()["content"][0]["text"]


## Run Command Function

Runs a terminal command on the local computer using subprocess.

A safety limit is applied to the output size â€” if the command returns 
more than 500 characters, it gets cut off. This prevents Claude from 
receiving overwhelming amounts of data and protects against token waste.

result.stdout â€” the normal output of the command
result.stderr â€” the error output if something went wrong

In [16]:
DANGEROUS_PATTERNS = [
    "format", "diskpart", "del /f", "del /s", "rmdir /s", "rd /s",
    "rm -rf", "rm -r", "shutdown /r", "shutdown /s", "bcdedit",
    "reg delete", "cipher /w", "remove-item -recurse", "format-volume",
    "clear-disk", "initialize-disk"
]

MAX_OUTPUT = 500

def is_dangerous(command):
    cmd_lower = command.lower()
    return any(pattern in cmd_lower for pattern in DANGEROUS_PATTERNS)

def run_command(command):
    if is_dangerous(command):
        return "BLOCKED: This command was identified as potentially destructive and was not executed."
    
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True
    )
    output = result.stdout + result.stderr
    
    if len(output) > MAX_OUTPUT:
        output = output[:MAX_OUTPUT] + "
[Output truncated at 500 characters]"
    
    return output


## Parse Reply Function

In Part 1 we parsed Claude's reply manually using string methods.
In Part 2 we use structured output â€” we instruct Claude to always 
reply in JSON format. This is more reliable and cleaner.

The function reads the JSON reply and extracts two things:
- type: either "command" or "answer"
- content: the actual command or answer text

If Claude returns invalid JSON for any reason, we catch the error 
and return "unknown" so the agent doesn't crash.

In [17]:
def parse_reply(reply):
    try:
        # Remove markdown code blocks if Claude added them
        clean = reply.strip()
        if clean.startswith("```"):
            clean = clean.split("```")[1]
            if clean.startswith("json"):
                clean = clean[4:]
        
        data = json.loads(clean.strip())
        return data["type"], data["content"]
    except:
        return "unknown", reply

## Run Agent Function â€” The ReAct Loop

This is the main brain of the agent. It takes the user's question 
and loops until Claude gives a final answer.

Upgrades from Part 1:
- Safety limit: maximum number of commands from config.json
- Session history: conversation is saved to a file after each message
- Tool output limit: commands cannot return more than 500 characters
- Structured output: replies are parsed as JSON instead of raw text

Each loop:
1. Send conversation history to Claude
2. Parse the JSON reply
3. If command â†’ check safety limit â†’ run it â†’ save to history â†’ loop again
4. If answer â†’ print it â†’ save to history â†’ stop

In [ ]:
def run_agent(user_input, messages):
    messages.append({"role": "user", "content": user_input})
    
    print(f"User: {user_input}")
    print("---")
    
    commands_run = 0
    max_commands = config["max_commands"]
    
    while True:
        reply = ask_claude(messages)
        print(f"Claude raw reply: {reply}")
        
        reply_type, content = parse_reply(reply)
        
        if reply_type == "command":
            if commands_run >= max_commands:
                print(f"Safety limit reached! Max {max_commands} commands allowed.")
                break
            print(f"Running: {content}")
            result = run_command(content)
            print(f"Result: {result}")
            commands_run += 1
            print(f"Commands used: {commands_run}/{max_commands}")
            
            messages.append({"role": "assistant", "content": reply})
            messages.append({"role": "user", "content": f"Command result: {result}"})
            save_history(messages)
            
        elif reply_type == "edit_file":
            parts = content.split("|||")
            if len(parts) == 3:
                filepath, old_text, new_text = parts
                result = edit_file(filepath, old_text, new_text)
                print(f"Edit result: {result}")
                commands_run += 1
                print(f"Commands used: {commands_run}/{max_commands}")
                messages.append({"role": "assistant", "content": reply})
                messages.append({"role": "user", "content": f"Edit result: {result}"})
                save_history(messages)
            else:
                print("Error: invalid edit_file format")
                break
            
        elif reply_type == "answer":
            print(f"Final answer: {content}")
            messages.append({"role": "assistant", "content": reply})
            save_history(messages)
            break
        
        else:
            print(f"Unexpected reply: {content}")
            break
            
    return messages

## Save History Function

In Part 1 the conversation history was lost when the program stopped.

In Part 2 we save the history to a JSON file after every message.

This means if you close and reopen the program, the conversation 
history is still there. This is called persistent storage.

The history is saved to a file called session_history.json in the 
same folder as the program.

In [19]:
HISTORY_FILE = "session_history.json"

def save_history(messages):
    with open(HISTORY_FILE, "w") as f:
        json.dump(messages, f, indent=2)

def load_history():
    try:
        with open(HISTORY_FILE, "r") as f:
            return json.load(f)
    except:
        return []

## Chat Loop

The main entry point of the agent. It loads the previous conversation 
history from the session file so the agent remembers past conversations.

The loop keeps asking for user input until the user types "quit".
Each question is passed to run_agent which handles the full ReAct loop.

Type "clear" to reset the conversation history and start fresh.
Type "quit" to exit the agent.

In [ ]:
# Load existing history or start fresh
messages = load_history()

if messages:
    print(f"Loaded {len(messages)} previous messages from history.")
else:
    print("Starting fresh conversation.")

print("Agent ready! Type 'quit' to exit or 'clear' to reset history.")
print("---")

while True:
    user_input = input("You: ")
    
    if user_input.lower() == "quit":
        print("Goodbye!")
        break
    
    if user_input.lower() == "clear":
        messages = []
        save_history(messages)
        print("History cleared!")
        continue
    
    if user_input.strip() == "":
        continue
    
    messages = run_agent(user_input, messages)
    print("---")

Loaded 14 previous messages from history.
Agent ready! Type 'quit' to exit or 'clear' to reset history.
---
User: what is my name
---
Claude raw reply: ```json
{
  "type": "answer",
  "content": "I don't have information about your name. You haven't provided it to me, and I don't have access to personal user information on your system. If you'd like to tell me your name, feel free to share it. Otherwise, I'm just here to help with your software engineering questions and tasks."
}
```
Claude raw reply: ```json
{
  "type": "answer",
  "content": "I don't have information about your name. You haven't provided it to me. If you'd like to tell me your name, feel free to share it. Otherwise, I'm here to assist you with software engineering tasks and questions."
}
```
Claude raw reply: ```json
{
  "type": "answer",
  "content": "I don't have information about your name. You haven't provided it to me, and I don't have access to personal user information on your system. If you'd like to tell me 

## File Editor Tool

This tool allows the agent to edit specific sections of files
without rewriting the entire file.

It works by finding an exact piece of text (old_text) in the file
and replacing it with new text (new_text).

This is safer than rewriting the whole file because:
- Only the targeted section changes
- The rest of the file stays untouched
- If the text is not found, it returns an error instead of breaking the file

In [ ]:
def edit_file(filepath, old_text, new_text):
    try:
        with open(filepath, "r") as f:
            content = f.read()
        
        if old_text not in content:
            return f"Error: could not find the text to replace in {filepath}"
        
        new_content = content.replace(old_text, new_text, 1)
        
        with open(filepath, "w") as f:
            f.write(new_content)
        
        return f"Successfully edited {filepath}"
    
    except Exception as e:
        return f"Error editing file: {str(e)}"